# Coherent transport with NEGForge

Follow a calculation from device geometry through electrostatics and charge feedback to states, transmission, the occupied transport window, and drain current. Every curve comes from the Rust-backed `negforge.Device`; the schematic only depicts the declared input geometry. Results illustrate a single longitudinal effective-mass mode, not a calibrated transistor.

From the repository root, after installing the updated extension in `.venv314`:
```bash
.venv314/bin/python scripts/render_demo.py
.venv314/bin/python -m jupyter notebook notebooks/negforge_demo.ipynb
.venv314/bin/python -m pytest -q -s tests/test_notebook.py
```
Use that environment as the notebook kernel. The source notebook intentionally contains no saved outputs. The [static gallery](../docs/figures/README.md) and its `sanity_checks.json` are regenerated by the same functions used below. NumPy, Matplotlib, Jupyter, and the native NEGForge extension are required. Coherent sections require `dev.transmission_spectrum(energies)`, coherent `dev.calc_current()`, and `dev.calc_thermionic_current()`.


In [ ]:
from pathlib import Path
import importlib.util
import json
import numpy as np
# Register rich figure formatters even in a fresh/headless kernel.
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats("png")
import matplotlib.pyplot as plt
from IPython.display import display

root = Path.cwd()
if not (root / "scripts/render_demo.py").exists():
    root = root.parent
assert (root / "scripts/render_demo.py").exists(), "Start in the repository or notebooks directory"
spec = importlib.util.spec_from_file_location("negforge_demo_figures", root / "scripts/render_demo.py")
demo = importlib.util.module_from_spec(spec)
spec.loader.exec_module(demo)

def show(fig):
    display(fig)
    plt.close(fig)

print(json.dumps(demo.PARAMS, indent=2))
print("Caller-supplied effective transverse area:", demo.AREA, "m²")


## 1. Geometry and decoupled electrostatics

The transport grid is 24 nm long: 8 nm source, 8 nm channel, and 8 nm drain, sampled at 0.25 nm (97 sites). `geo=1` sets the natural-length electrostatic reduction; the drawing is not a resolved 2D Poisson solution. The caller explicitly supplies **20 nm²** for the spin-summed volume charge. It is not derived from the 5 nm body thickness, and it does not add transverse modes or multiply single-mode current by area.

`psi_f` is electron potential energy **in eV**, not electric voltage. The reservoir chemical potentials are 0.05 eV and −0.05 eV at `Vds=0.1 V`. Finite, short screened contact regions mean the calculated endpoint band edges are not pinned to those chemical potentials. Transport attaches leads matching those actual endpoint energies.


In [ ]:
dec, sc, fine, eta_device, checks = demo.electrostatic_examples()
show(demo.geometry_figure(dec))
print("Screening length (nm):", checks["screening_length_nm"])
print("Actual endpoint band energies (eV):", dec.psi_f[[0, -1]])


## 2. Electrostatic vs finite-area self-consistency

The decoupled calculation has zero mobile charge. The Poisson–NEGF calculation feeds **spin-summed negative electron charge** back into Poisson using the explicit area. Repulsion raises electron potential energy. The implicit unit-area normalization is also checked numerically; it does not describe a physical cross-section and barely changes the potential.

The stopping residual is in eV. We separately halve the energy step at fixed `eta`, then change both step and `eta`. A small iterative residual does **not** establish integration accuracy or the `eta → 0` limit. Here eta is an absorbing finite-device regularization without matching particle reinjection; its effect is not tunneling, scattering, or a conserving dephasing model.


In [ ]:
show(demo.feedback_figure(dec, sc, fine, eta_device))
for key in ("scf_iterations", "scf_residual_eV", "decoupled_barrier_eV",
            "scf_barrier_eV", "max_feedback_eV", "peak_abs_charge_C_m3",
            "implicit_unit_area_max_feedback_eV", "fixed_eta_half_dE_barrier_eV",
            "half_eta_half_dE_barrier_eV"):
    print(f"{key}: {checks[key]:.9g}")


## 3. LDOS: available states, not transmitted flux

The API returns `−Im Gᵣᵢᵢ/a`; divide by π for **per-spin LDOS in (eV nm)⁻¹**. The two panels use the same logarithmic color scale and energy range; cyan overlays the respective potential and white marks the source chemical potential. The view zooms to 0.25 eV to show the barrier region, while the full computed LDOS arrays extend farther. The visual saturation cap is the larger 99.5th percentile of the two arrays.

A finite LDOS below a barrier is not a transmission probability and does not by itself establish current. The LDOS helper uses its fixed retarded broadening; SCF uses the separately declared eta. Narrow spectral features need energy-grid checks rather than an interpretation as flux.


In [ ]:
show(demo.ldos_figure(dec, sc, checks))
print(json.dumps(checks["ldos"], indent=2))


## 4. Transmission, Fermi window, and spectral current

`transmission_spectrum(energies)` supplies the coherent two-lead **T(E)**. Dotted lines mark the barrier maximum for each potential. Transmission below that maximum, **inside both propagating lead bands**, is evidence of sub-barrier coherent transport. No propagating source/drain channel means no two-terminal transmission, even if local spectral weight exists.

The occupied transport window is `f_source − f_drain`. The spectral current is

$$j(E)=\frac{2e^2}{h} T(E)[f_s(E)-f_d(E)]\quad\mathrm{A/eV},\qquad I=\int j(E)\,dE.$$

The extra e in the prefactor converts the numerical eV integration measure to joules. The factor two counts spin once. Filled spectral-current regions are below each device's own barrier; their areas, not LDOS intensity, quantify the sub-barrier current. The script integrates a midpoint energy grid inside the common propagating band, checks T bounds, halves the step, and compares against `calc_current()` (3% guard, actual mismatch reported).


In [ ]:
demo.require_transport()  # Fails clearly on an older installed wheel; no surrogate data.
show(demo.transport_figure(dec, sc, checks))
print(json.dumps(checks["spectral_current"], indent=2))


## 5. Final transfer and output characteristics

At every bias, start from a fresh device so sweeps do not inherit hidden charge or solver history. Both electrostatic and finite-area SCF potentials are evaluated with coherent current and the explicitly named thermionic comparator. Solid curves use `calc_current()`; dashed curves use `calc_thermionic_current()` with step transmission `Θ(E − max U)`.

Their difference is **not exclusively tunneling**: coherent transport includes reflection and finite lead bandwidth, whereas the thermionic approximation has an ideal above-barrier channel. The output sweep includes equilibrium, where both currents must vanish. The transfer plot uses a log scale; no broad-range subthreshold-swing fit is claimed. Solver failures raise errors instead of silently omitting points.


In [ ]:
show(demo.iv_figure(checks))
for row in checks["iv_points"]:
    print(f"{row['sweep']}={row['voltage_V']:.3f} V | SCF={row['scf']} | "
          f"coherent={row['coherent_A']:.6e} A | thermionic={row['thermionic_A']:.6e} A | "
          f"residual={row['residual_eV']}")


## Interpretation and reproducibility

The finite-area result is a specified toy single-mode device, not a quantitative silicon prediction. Natural-length electrostatics, the finite-difference band, finite contacts, eta-dependent injected charge, missing transverse/valley structure, and unassigned isolated bound-state occupations limit the model. Small SCF residuals do not remove these limitations. Inspect the reported eta sensitivity before interpreting changes in current.

Run `.venv314/bin/python scripts/render_demo.py` to regenerate all five static figures plus numeric assertions in `docs/figures/sanity_checks.json`. `--electrostatics-only` explicitly renders just geometry, SCF, and LDOS when the updated transport extension is not installed; the JSON then records `transport_rendered: false`. Keep notebook outputs cleared when saving the source.

See [physics and units](../docs/PHYSICS.md) and [explicit-area electrostatics](../docs/ELECTROSTATIC_GEOMETRY.md).
